# GeoDeep YOLOv9 / DeepForest → SAM 2.1: несколько ортофотопланов

Пакетная версия `geodeep_yolov9_sam`: все ортофотопланы из папки `SRC_DIR` обрабатываются по очереди одинаковыми параметрами.

Для каждого файла:
1. ортофото перепроецируется в UTM и приводится к `RES_M` (копия в `PREP_DIR`, повторно не пересчитывается);
2. **GeoDeep YOLOv9 → SAM 2.1** (`geodeep_sam`) и **DeepForest → SAM 2.1** (`deepforest_sam`);
3. результаты сохраняются в `OUT_ROOT/<имя файла>/`; что именно сохранять, задаётся словарём `SAVE` в ячейке 4.

В конце строится общая таблица по всем файлам: `OUT_ROOT/all_files_summary.csv`.

Порядок: ячейки 1–9, 11–16 только объявляют функции и выполняются быстро. Ячейка 10 (необязательно) подбирает расширение рамок на одном файле. Вся обработка запускается ячейкой 17.

## Подготовка

In [ ]:
# @title 1. Установка библиотек
!pip install -q ultralytics rasterio geopandas shapely matplotlib tqdm scikit-image geodeep deepforest
# Если после установки импорт torch или numpy выдаёт ошибку, перезапустите среду (Runtime → Restart) и продолжите с ячейки 2

In [ ]:
# @title 2. Подключение Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# @title 3. Подготовка ортофото (перепроецирование + изменение разрешения)
import os
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, transform_bounds
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window
from rasterio.enums import Resampling, ColorInterp


def utm_epsg_for(src):
    # EPSG UTM-зоны по центру растра
    l, b, r, t = transform_bounds(src.crs, 'EPSG:4326', *src.bounds)
    lon, lat = (l + r) / 2, (b + t) / 2
    zone = int((lon + 180) // 6) + 1
    return (32600 if lat >= 0 else 32700) + zone


def rgb_bands(src):
    # Номера каналов R, G, B: по разметке каналов в файле, иначе первые три
    ci = list(src.colorinterp)
    if all(c in ci for c in (ColorInterp.red, ColorInterp.green, ColorInterp.blue)):
        return [ci.index(ColorInterp.red) + 1, ci.index(ColorInterp.green) + 1, ci.index(ColorInterp.blue) + 1]
    if src.count < 3:
        raise ValueError(f'В файле {src.count} канал(а), нужен RGB')
    return [1, 2, 3]


def stretch_params(src, bands, max_side=2000):
    # Для не-uint8 растров: общие для R, G, B границы растяжки в 0..255 (0.5 и 99.5 перцентили),
    # общие, чтобы не исказить цвета
    if src.dtypes[bands[0] - 1] == 'uint8':
        return None
    sc = min(1.0, max_side / max(src.width, src.height))
    shape = (max(1, int(src.height * sc)), max(1, int(src.width * sc)))
    m = src.dataset_mask(out_shape=shape) > 0
    a = src.read(bands, out_shape=(len(bands),) + shape, masked=False).astype('float64')
    v = a[:, m & np.isfinite(a).all(0)]
    if v.size == 0:
        return 0.0, 255.0
    lo, hi = np.percentile(v, [0.5, 99.5])
    return float(lo), float(hi) if hi > lo else float(lo) + 1.0


def has_own_mask(src):
    # Есть ли у исходника nodata, альфа-канал или маска (иначе пустые поля по краю обычно просто чёрные)
    from rasterio.enums import MaskFlags
    return src.nodata is not None or any(MaskFlags.all_valid not in f for f in src.mask_flag_enums)


def prepare_ortho(src_path, dst_path, res=0.1, dst_crs=None,
                  resampling=Resampling.average, overwrite=False, block=2048):
    # Перепроецирует ортофото в метровую CRS, приводит к размеру пикселя res (м) и к 8-битному RGB.
    # Пустые области (nodata, альфа или маска исходника; если их нет — чисто чёрный фон) записываются
    # в 4-й канал-альфу, чтобы модели и метрики их не учитывали.
    # dst_crs=None -> UTM-зона подбирается автоматически
    if os.path.exists(dst_path) and not overwrite:
        print('Уже существует, пропускаю:', dst_path)
        return dst_path

    with rasterio.open(src_path) as src:
        if src.crs is None:
            raise ValueError(f'У файла нет системы координат: {src_path}')
        if dst_crs is None:
            dst_crs = f'EPSG:{utm_epsg_for(src)}'
        bands = rgb_bands(src)
        stretch = stretch_params(src, bands)
        black_is_empty = not has_own_mask(src)   # нет nodata/альфы -> чисто чёрные пиксели считаем пустыми
        transform, w, h = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds, resolution=res)
        print(f'{src.crs} -> {dst_crs}, {src.width}x{src.height} -> {w}x{h}, пиксель {res} м | '
              f'каналов {src.count} ({src.dtypes[0]}), RGB = {bands}, nodata = {src.nodata}'
              + (f' | растяжка {stretch[0]:.1f}..{stretch[1]:.1f} -> 0..255' if stretch else '')
              + (' | маски нет: чёрный фон = пусто' if black_is_empty else ''))

        profile = dict(driver='GTiff', crs=dst_crs, transform=transform, width=w, height=h,
                       count=4, dtype='uint8', nodata=None, compress='deflate',
                       tiled=True, blockxsize=256, blockysize=256, BIGTIFF='IF_SAFER')
        with WarpedVRT(src, crs=dst_crs, transform=transform, width=w, height=h,
                       resampling=resampling) as vrt, rasterio.open(dst_path, 'w', **profile) as dst:
            for y in range(0, h, block):
                for x in range(0, w, block):
                    win = Window(x, y, min(block, w - x), min(block, h - y))
                    mask = vrt.dataset_mask(window=win) > 0
                    data = vrt.read(bands, window=win).astype('float32')
                    if stretch:
                        lo, hi = stretch
                        data = (data - lo) / (hi - lo) * 255
                    data = np.nan_to_num(np.clip(data, 0, 255)).astype('uint8')
                    if black_is_empty:
                        mask &= data.max(0) > 0
                    data[:, ~mask] = 0
                    dst.write(data, [1, 2, 3], window=win)
                    dst.write((mask * 255).astype('uint8'), 4, window=win)
            dst.colorinterp = [ColorInterp.red, ColorInterp.green, ColorInterp.blue, ColorInterp.alpha]
    return dst_path

In [ ]:
# @title 4. Параметры
import os, glob, numpy as np, rasterio

# Меньше фрагментация памяти GPU; действует, только если ячейка выполнена до первой загрузки модели в сессии
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
# ============ ПУТИ ============
BASE_DIR  = "/content/drive/MyDrive/tree_model_test"
SRC_DIR   = f"{BASE_DIR}/ortho_in"           # папка с исходными ортофотопланами
FILE_MASK = "*.tif"                          # какие файлы брать из SRC_DIR (например "*сосняк*.tif")
PREP_DIR  = f"{BASE_DIR}/ortho_prepared"     # подготовленные копии (UTM, RES_M); не должна совпадать с SRC_DIR
OUT_ROOT  = f"{BASE_DIR}/results_multi"      # для каждого файла создаётся подпапка с его именем

# Эталонная разметка (необязательно): для файла X.tif ищутся REF_DIR/X_ref_crowns.gpkg и REF_DIR/X_ref_plots.gpkg
REF_DIR = f"{BASE_DIR}/reference"

# ============ ПОДГОТОВКА ОРТОФОТО ============
RES_M     = 0.10      # целевой размер пикселя, м
DST_CRS   = None      # None = UTM подберётся по каждому файлу, либо 'EPSG:32636'
OVERWRITE = False     # True = пересоздать подготовленные растры (после смены RES_M)

# ============ ТАЙЛИНГ (для SAM) ============
TILE     = 1024       # размер окна, пикс
OVERLAP  = 256        # перекрытие, пикс. Должно быть >= диаметра крупной кроны в пикселях
ROI_USER = None       # None = каждый снимок целиком; (col_off, row_off, width, height) = одно окно для всех файлов

# ============ ФИЛЬТРЫ КРОН ============
MIN_AREA_M2 = 2.0     # меньше = мусор
MAX_AREA_M2 = 80.0    # больше = скорее всего несколько слипшихся крон

# ============ КАКИЕ СВЯЗКИ ЗАПУСКАТЬ ============
RUN_YOLO_SAM = True   # GeoDeep YOLOv9 -> SAM 2.1
RUN_DF_SAM   = False   # DeepForest -> SAM 2.1
MODEL_KEY    = 'geodeep_sam'
RN_KEY       = 'deepforest_sam'

# ============ GeoDeep YOLOv9 (рамки) ============
GEODEEP_MODEL       = 'trees_yolov9'
GEODEEP_MIN_VISIBLE = 0.5              # рамку на краю тайла берём, если видно >=50% её площади

# Расширение рамок YOLOv9 перед SAM (рамки часто охватывают только верхушку кроны).
# Новая сторона = сторона * BOX_SCALE + 2 * BOX_PAD_M, затем не меньше BOX_MIN_SIDE_M
BOX_SCALE      = 1.2    # во сколько раз увеличить сторону рамки (1.1-1.4)
BOX_PAD_M      = 0.0    # добавить с каждой стороны, м
BOX_MIN_SIDE_M = 0.0    # минимальная сторона рамки, м (0 = не трогать)
BOX_SQUARE     = False  # True = делать рамку квадратной по длинной стороне

# ============ DeepForest (рамки) ============
DF_MODEL   = 'weecology/deepforest-tree'
DF_SCORE   = 0.1      # порог уверенности; выше = меньше ложных рамок
DF_PATCH   = 400      # окно, пикс: DeepForest обучен на окнах 400 пикс при ~10 см/пикс
DF_OVERLAP = 0.25     # перекрытие окон, доля
RN_BOX     = (1.0, 0.0, 0.0, False)   # расширение рамок DeepForest: (SCALE, PAD_M, MIN_SIDE_M, SQUARE)

# ============ SAM 2 ============
SAM_WEIGHTS       = 'sam2.1_b.pt'   # sam2.1_t / s / b / l: больше = точнее, но медленнее
SAM_IMGSZ         = 1024
SAM_BOX_MIN_GREEN = 0.0             # 0 = не фильтровать по ExG (рамка уже указывает на дерево)
SAM_NMS_THR       = 0.5             # маска отбрасывается, если >50% её площади уже занято другой маской
SAM_MAX_BOXES     = 64              # сколько рамок подавать в SAM за один вызов; меньше = меньше памяти GPU, но медленнее

# ============ ОЦЕНКА ПО ЭТАЛОНУ ============
IOU_THR = 0.5

# ============ ЧТО СОХРАНЯТЬ ДЛЯ КАЖДОГО ФАЙЛА ============
SAVE = dict(
    crowns_gpkg  = True,  # <связка>.gpkg: только контуры и score (то же, что в *_review.gpkg, но без атрибутов)
    review_gpkg  = False,   # <связка>_review.gpkg: контуры + площадь, диаметр, keep/reason — основной результат
    boxes_gpkg   = False,  # рамки детекторов *_boxes_raw.gpkg и *_boxes.gpkg (для диагностики в QGIS)
    summary_csv  = False,  # model_summary.csv в папке файла (те же строки есть в all_files_summary.csv)
    summary_html = False,  # model_summary.html (копия csv)
    closeup_png  = False,   # visual_closeup.png: крупный план 60x60 м
    full_png     = True,   # full_<связка>.png: весь снимок с контурами
    keep_prepared = True,  # False = удалять подготовленную копию ортофото (PREP_DIR) после обработки файла
)
assert SAVE['crowns_gpkg'] or SAVE['review_gpkg'], 'нужно сохранять хотя бы один файл с контурами'

# ============ ВИЗУАЛИЗАЦИЯ ============
LINE_COLOR = (0, 90, 255)     # цвет границ крон, RGB (синий)
HALO_COLOR = (255, 255, 255)  # белая обводка вокруг линии; None = без неё
BOX_COLOR  = (255, 140, 0)    # цвет рамок на картинке крупного плана
LINE_PX    = 3
HALO_PX    = 1
VIS_SIZE_M = 60               # сторона квадрата для крупного плана, м (берётся центр каждого снимка)

# ============ ПРОИЗВОДНЫЕ ЗНАЧЕНИЯ (не редактировать) ============
LINE_HEX = '#%02x%02x%02x' % LINE_COLOR
HALO_HEX = '#%02x%02x%02x' % HALO_COLOR if HALO_COLOR else None
BOX_HEX  = '#%02x%02x%02x' % BOX_COLOR
os.makedirs(PREP_DIR, exist_ok=True)
os.makedirs(OUT_ROOT, exist_ok=True)
assert os.path.abspath(PREP_DIR) != os.path.abspath(SRC_DIR), 'PREP_DIR не должна совпадать с SRC_DIR'

FILE_MASK = "*.tif"                          # какие файлы брать из SRC_DIR (например "*сосняк*.tif")
# Выбор файлов для прогона: имена из SRC_DIR (с .tif или без), можно с * и ?
ONLY_FILES = ["Ортофотоплан (Бульвар Победы)"]                              # [] = все по FILE_MASK; например ['орто_сосняк', 'бульвар_победы.tif']
SKIP_FILES = []                              # исключить, например ['орто 9 км']

import fnmatch, unicodedata


def _norm(s):
    # Нижний регистр + единая форма Unicode (на Диске «й», «ё» в именах бывают записаны двумя символами)
    return unicodedata.normalize('NFC', s).lower()


def _match(path, patterns):
    # Совпадает ли имя файла с одним из шаблонов (регистр не важен, расширение можно не писать)
    name = _norm(os.path.basename(path))
    stem = os.path.splitext(name)[0]
    return any(fnmatch.fnmatch(name, _norm(p)) or fnmatch.fnmatch(stem, _norm(p)) for p in patterns)


ALL_FILES = sorted(glob.glob(os.path.join(SRC_DIR, FILE_MASK)))
FILES = [f for f in ALL_FILES
         if (not ONLY_FILES or _match(f, ONLY_FILES)) and not _match(f, SKIP_FILES)]
for p in ONLY_FILES:
    if not any(_match(f, [p]) for f in ALL_FILES):
        print(f'ВНИМАНИЕ: в ONLY_FILES указан "{p}", но такого файла в {SRC_DIR} нет')
print(f'Найдено файлов: {len(ALL_FILES)}, к обработке: {len(FILES)}')
for f in ALL_FILES:
    print(' ', '+' if f in FILES else '-', os.path.basename(f))


def set_file(src_path):
    # Готовит ортофото и выставляет глобальные параметры текущего файла
    global ORTHO, OUT, GSD, ROI, MIN_AREA_PX, MAX_AREA_PX, FILE_NAME
    FILE_NAME = os.path.splitext(os.path.basename(src_path))[0]
    ORTHO = prepare_ortho(src_path, f'{PREP_DIR}/{FILE_NAME}_utm_{round(RES_M * 100)}cm.tif',
                          res=RES_M, dst_crs=DST_CRS, overwrite=OVERWRITE)
    OUT = f'{OUT_ROOT}/{FILE_NAME}'
    os.makedirs(OUT, exist_ok=True)
    with rasterio.open(ORTHO) as s:
        print(s.crs, s.res, s.width, s.height, s.count, s.dtypes)
        if s.crs is None or s.crs.is_geographic:
            print('ВНИМАНИЕ: CRS в градусах, площади будут неверными. Проверьте prepare_ortho.')
        GSD = s.res[0]
        ROI = ROI_USER or (0, 0, s.width, s.height)
    MIN_AREA_PX = max(1, int(round(MIN_AREA_M2 / GSD**2)))
    MAX_AREA_PX = int(round(MAX_AREA_M2 / GSD**2))
    print(f'{FILE_NAME}: GSD={GSD:.3f} м | площадь кроны {MIN_AREA_PX}-{MAX_AREA_PX} пикс | результаты: {OUT}')

Найдено файлов: 6, к обработке: 1
  + Ортофотоплан (Бульвар Победы).tif
  - Ортофотоплан (Кивач, ООПТ).tif
  - Ортофотоплан (Пп ельник).tif
  - Ортофотоплан (Пп смешанный лес).tif
  - Ортофотоплан (пп березняк).tif
  - орто_сосняк.tif


In [ ]:
# @title 5. Тайлинг
import time
import geopandas as gpd, shapely
from rasterio.windows import Window
from rasterio.features import rasterize
from affine import Affine
from shapely.geometry import Polygon, box
from shapely.affinity import affine_transform
from tqdm.auto import tqdm


def iter_tiles(roi, tile, overlap):
    x0, y0, W, H = roi
    step = tile - overlap
    for y in range(y0, y0 + H, step):
        if y > y0 and y + overlap >= y0 + H: break      # хвост уже покрыт предыдущим рядом
        for x in range(x0, x0 + W, step):
            if x > x0 and x + overlap >= x0 + W: break
            yield Window(x, y, min(tile, x0 + W - x), min(tile, y0 + H - y))


def read_tile(src, win):
    arr = src.read([1, 2, 3], window=win)
    if arr.dtype != np.uint8:
        arr = np.clip(arr.astype(np.float32) / arr.max() * 255 if arr.max() > 255 else arr, 0, 255).astype(np.uint8)
    valid = src.dataset_mask(window=win)
    if valid.mean() < 10:        # почти пустой тайл (вне ортофото)
        return None
    return np.ascontiguousarray(arr.transpose(1, 2, 0))

In [ ]:
# @title 6. Универсальный прогон модели
# predict_fn(img, win) -> [(контур Nx2 в пикселях тайла, score), ...]
def run_model(name, predict_fn, roi=None, tile=None, overlap=None, extra_time=0.0):
    roi = roi or ROI
    tile = tile or TILE
    overlap = OVERLAP if overlap is None else overlap
    recs, t0 = [], time.time()
    x0, y0, W, H = roi
    with rasterio.open(ORTHO) as src:
        crs = src.crs
        for win in tqdm(list(iter_tiles(roi, tile, overlap)), desc=name):
            img = read_tile(src, win)
            if img is None:
                continue
            m = overlap // 2     # по краю растра отступ не нужен
            lo = 0 if win.col_off == x0 else m
            to = 0 if win.row_off == y0 else m
            ro = 0 if win.col_off + win.width  >= x0 + W else m
            bo = 0 if win.row_off + win.height >= y0 + H else m
            core = box(lo, to, win.width - ro, win.height - bo)
            wt = src.window_transform(win)
            mat = [wt.a, wt.b, wt.d, wt.e, wt.c, wt.f]
            for pts, score in predict_fn(img, win):
                pts = np.asarray(pts, dtype=float)
                if len(pts) < 3: continue
                p = Polygon(pts)
                if not p.is_valid: p = p.buffer(0)
                if p.geom_type == 'MultiPolygon':         # после buffer(0) берём крупнейшую часть
                    p = max(p.geoms, key=lambda q: q.area)
                if p.is_empty or not core.contains(p.centroid):   # дедупликация зон перекрытия
                    continue
                recs.append({'score': float(score), 'geometry': affine_transform(p, mat)})
    if recs:
        gdf = gpd.GeoDataFrame(recs, geometry='geometry', crs=crs)
    else:
        gdf = gpd.GeoDataFrame({'score': [], 'geometry': []}, geometry='geometry', crs=crs)
    gdf.attrs['time_s'] = time.time() - t0 + extra_time
    if SAVE['crowns_gpkg'] and not name.startswith('sweep_'):   # прогоны подбора рамок не сохраняем
        gdf.to_file(f'{OUT}/{name}.gpkg', driver='GPKG')
    print(f'{name}: {len(gdf)} крон, {gdf.attrs["time_s"]:.0f} с')
    return gdf

In [ ]:
# @title 7. Вспомогательные функции и метрики без разметки
import cv2
import pandas as pd
from rasterio.enums import Resampling
from skimage.filters import threshold_otsu


def mask_to_polys(mask, score=1.0, min_area=30, largest=False):
    # Бинарная маска -> контуры. largest=True: одна маска = одна крона, берём крупнейший контур
    cnts, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts = [c for c in cnts if cv2.contourArea(c) >= min_area]
    if largest and cnts:
        cnts = [max(cnts, key=cv2.contourArea)]
    return [(c[:, 0, :], score) for c in cnts]


def poly_patch(pts, shape):
    # Растеризует контур в пределах его рамки: (y0, y1, x0, x1, bool-маска) или None
    H, W = shape
    p = np.asarray(pts).astype(np.int32)
    bx, by, bw, bh = cv2.boundingRect(p)
    x0, y0, x1, y1 = max(bx, 0), max(by, 0), min(bx + bw, W), min(by + bh, H)
    if x1 <= x0 or y1 <= y0:
        return None
    m = np.zeros((y1 - y0, x1 - x0), np.uint8)
    cv2.fillPoly(m, [(p - [x0, y0]).astype(np.int32)], 1)
    return y0, y1, x0, x1, m.astype(bool)


def tile_exg(img):
    f = img.astype(np.float32)
    return (2 * f[..., 1] - f[..., 0] - f[..., 2]) / (f.sum(2) + 1e-6)


def green_frac(pts, green):
    pp = poly_patch(pts, green.shape)
    if pp is None: return 0.0
    y0, y1, x0, x1, m = pp
    g = green[y0:y1, x0:x1][m]
    return float(g.mean()) if g.size else 0.0


def suppress_overlaps(items, shape, thr=0.5):
    # Идём от высокого score к низкому; маску отбрасываем, если больше thr её площади уже занято
    canvas = np.zeros(shape, bool)
    keep = []
    for pts, s in sorted(items, key=lambda t: -t[1]):
        pp = poly_patch(pts, shape)
        if pp is None: continue
        y0, y1, x0, x1, m = pp
        n = m.sum()
        if n == 0: continue
        if (canvas[y0:y1, x0:x1] & m).sum() / n > thr:
            continue
        canvas[y0:y1, x0:x1] |= m
        keep.append((pts, s))
    return keep


def ultra_masks_to_items(r, img_shape, scores):
    # Маски Ultralytics -> [(контур, score)], по одному крупнейшему контуру на маску
    out = []
    data = r.masks.data.cpu().numpy() if r.masks is not None else []
    if len(data) and data.shape[1:] == tuple(img_shape):
        for m, s in zip(data, scores):
            out.extend(mask_to_polys(m > 0.5, float(s), MIN_AREA_PX, largest=True))
    elif r.masks is not None:                       # маски в другом масштабе -> готовые контуры
        for pts, s in zip(r.masks.xy, scores):
            if len(pts) >= 3:
                out.append((pts, float(s)))
    return out


def read_overview(roi, max_side=3000):
    x0, y0, W, H = roi
    sc = min(1.0, max_side / max(W, H))
    ow, oh = int(W * sc), int(H * sc)
    with rasterio.open(ORTHO) as src:
        win = Window(x0, y0, W, H)
        arr = src.read([1, 2, 3], window=win, out_shape=(3, oh, ow),
                       resampling=Resampling.average).astype(np.float32)
        tr = src.window_transform(win) * Affine.scale(W / ow, H / oh)
        valid = src.read_masks(1, window=win, out_shape=(oh, ow)) > 0
    return arr, tr, valid


def set_overview():
    # Обзорный растр, ExG и маска «зелёного» для текущего файла
    global arr, tr, valid, exg, veg, EXG_THR, _mask_cache
    arr, tr, valid = read_overview(ROI)
    if not valid.any():
      raise ValueError('На снимке не осталось непустых пикселей (маска/альфа пустая). '
                      'Пересоздайте подготовленный файл: OVERWRITE = True в ячейке 4')
    R, G, B = arr; s_ = R + G + B + 1e-6
    exg = 2*G/s_ - R/s_ - B/s_                       # Excess Green
    EXG_THR = float(threshold_otsu(exg[valid]))
    veg = (exg > EXG_THR) & valid                    # грубая маска «зелёного»
    _mask_cache = {}                                 # маски прошлого файла больше не нужны
    print('Порог ExG (Оцу):', round(EXG_THR, 4))


_mask_cache = {}


def cached_mask(name, gdf):
    # Растровая маска полигонов в системе обзорного растра
    if name not in _mask_cache:
        if len(gdf) == 0:
            _mask_cache[name] = np.zeros(exg.shape, bool)
        else:
            _mask_cache[name] = rasterize(((g, 1) for g in gdf.geometry), out_shape=exg.shape,
                                          transform=tr, fill=0).astype(bool) & valid
    return _mask_cache[name]


def agreement(na, a, nb, b):
    # IoU покрытия двух моделей (по растровым маскам)
    ma, mb = cached_mask(na, a), cached_mask(nb, b)
    return float((ma & mb).sum() / max((ma | mb).sum(), 1))


def proxy_metrics(name, gdf):
    m = cached_mask(name, gdf)
    geoms = gdf.geometry
    area = geoms.area
    compact = 4 * np.pi * area / geoms.length**2      # 1 = круг, мало = рваная/вытянутая форма
    union_area = geoms.union_all().area if hasattr(geoms, 'union_all') else geoms.unary_union.area
    return {
        'n_crowns': len(gdf),
        'time_s': round(gdf.attrs.get('time_s', np.nan)) if np.isfinite(gdf.attrs.get('time_s', np.nan)) else np.nan,
        'median_area': round(area.median(), 2),
        'p5_area': round(area.quantile(.05), 2),
        'p95_area': round(area.quantile(.95), 2),
        'giant_pct': round(100 * (area > MAX_AREA_M2).mean(), 1),
        'compact_med': round(compact.median(), 3),
        'self_overlap': round(area.sum() / union_area - 1, 3),
        'cover_pct': round(100 * m.sum() / valid.sum(), 1),
        'exg_in': round(float(exg[m].mean()), 3),
        'exg_out': round(float(exg[~m & valid].mean()), 3),
        'veg_in_poly_pct': round(100 * (veg & m).sum() / max(m.sum(), 1), 1),
        'veg_covered_pct': round(100 * (veg & m).sum() / max(veg.sum(), 1), 1),
    }

results = {}

## Связки «детектор → SAM 2.1»

In [ ]:
# @title 8. GeoDeep YOLOv9: рамки (функции)
from geodeep import detect


def geodeep_boxes(model=GEODEEP_MODEL, roi=None):
    # Рамки GeoDeep по всему ROI в пикселях ORTHO: (boxes Nx4, scores N, время, с)
    roi = roi or ROI
    t0 = time.time()
    x0, y0, W, H = roi
    with rasterio.open(ORTHO) as src:
        if (W, H) == (src.width, src.height):
            path = ORTHO
        else:                                      # вырезаем ROI во временный GeoTIFF
            win = Window(x0, y0, W, H)
            path = '/content/roi_tmp.tif'
            prof = src.profile; prof.update(width=W, height=H, transform=src.window_transform(win), count=3)
            with rasterio.open(path, 'w', **prof) as dst:
                dst.write(src.read([1, 2, 3], window=win))
    bboxes, scores, classes = detect(path, model)
    bboxes = np.asarray(bboxes, dtype=float).reshape(-1, 4)
    assert bboxes.size == 0 or (bboxes[:, [0, 2]].max() <= W + 1 and bboxes[:, [1, 3]].max() <= H + 1), \
        'координаты боксов не похожи на пиксели растра; проверьте формат ответа geodeep'
    bboxes += [x0, y0, x0, y0]
    print(f'GeoDeep: {len(bboxes)} рамок, {time.time() - t0:.0f} с')
    return bboxes, np.asarray(scores, dtype=float), time.time() - t0


def boxes_to_gdf(boxes, scores):
    # Рамки в пикселях ORTHO -> GeoDataFrame в координатах ORTHO
    with rasterio.open(ORTHO) as src:
        t, crs = src.transform, src.crs
    mat = [t.a, t.b, t.d, t.e, t.c, t.f]
    geoms = [affine_transform(box(x1, y1, x2, y2), mat) for x1, y1, x2, y2 in boxes]
    return gpd.GeoDataFrame({'score': scores}, geometry=geoms, crs=crs)


def expand_boxes(boxes, scale=1.0, pad_m=0.0, min_side_m=0.0, square=False):
    # Расширяет рамки от центра (пиксели ORTHO) и обрезает по границам растра
    b = np.asarray(boxes, dtype=float).reshape(-1, 4)
    if len(b) == 0:
        return b
    cx, cy = (b[:, 0] + b[:, 2]) / 2, (b[:, 1] + b[:, 3]) / 2
    w, h = b[:, 2] - b[:, 0], b[:, 3] - b[:, 1]
    if square:
        w = h = np.maximum(w, h)
    pad, mn = pad_m / GSD, min_side_m / GSD
    w = np.maximum(w * scale + 2 * pad, mn)
    h = np.maximum(h * scale + 2 * pad, mn)
    with rasterio.open(ORTHO) as src:
        W, H = src.width, src.height
    return np.clip(np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], 1), 0, [W, H, W, H])


side = lambda b: np.sqrt((b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])) * GSD   # сторона рамки, м

In [ ]:
# @title 9. SAM 2.1 по рамкам GeoDeep (функции)
import matplotlib.pyplot as plt
from ultralytics import SAM
import torch

_SAM = None
def get_sam():
    global _SAM
    if _SAM is None:
        _SAM = SAM(SAM_WEIGHTS)                # веса скачаются автоматически
    return _SAM


def box_sam(img, boxes, scores):
    # Одна маска SAM на каждую рамку (x1, y1, x2, y2 в пикселях тайла).
    # Рамки подаются пачками по SAM_MAX_BOXES: в плотном лесу в тайле бывает несколько сотен рамок,
    # и все сразу не помещаются в память GPU. При нехватке памяти пачка автоматически уменьшается вдвое
    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4)
    scores = np.asarray(scores, dtype=float)
    if len(boxes) == 0:
        return []
    bgr = np.ascontiguousarray(img[..., ::-1])
    items, i, step = [], 0, SAM_MAX_BOXES
    while i < len(boxes):
        b, s = boxes[i:i + step], scores[i:i + step]
        try:
            r = get_sam().predict(bgr, bboxes=b.tolist(), imgsz=SAM_IMGSZ, verbose=False)[0]
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if step == 1:
                raise
            step = max(1, step // 2)
            print(f'Не хватило памяти GPU, пачка рамок уменьшена до {step}')
            continue
        if r.masks is not None:
            items.extend(ultra_masks_to_items(r, img.shape[:2], s))
        del r
        i += len(b)
    if SAM_BOX_MIN_GREEN > 0:
        green = tile_exg(img) > EXG_THR
        items = [(p, s) for p, s in items if green_frac(p, green) >= SAM_BOX_MIN_GREEN]
    items = [(p, s) for p, s in items
             if MIN_AREA_PX <= cv2.contourArea(np.asarray(p, np.float32)) <= MAX_AREA_PX]
    return suppress_overlaps(items, img.shape[:2], SAM_NMS_THR)


def make_geodeep_sam_fn(boxes, scores):
    def fn(img, win):
        h, w = img.shape[:2]
        b = boxes - [win.col_off, win.row_off, win.col_off, win.row_off]
        sel = (b[:, 2] > 0) & (b[:, 0] < w) & (b[:, 3] > 0) & (b[:, 1] < h)
        b, sc = b[sel], scores[sel]
        if len(b) == 0:
            return []
        bc = np.clip(b, 0, [w, h, w, h])
        a0 = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
        a1 = (bc[:, 2] - bc[:, 0]) * (bc[:, 3] - bc[:, 1])
        keep = a1 >= GEODEEP_MIN_VISIBLE * np.maximum(a0, 1e-6)
        return box_sam(img, bc[keep], sc[keep])
    return fn


def box_hit_pct(polys, boxes_gdf, tol_m=None):
    # % крон, упёршихся в свою рамку хотя бы одной стороной: высокий процент = рамки малы и режут крону
    tol = tol_m if tol_m is not None else 1.5 * GSD
    if len(polys) == 0 or len(boxes_gdf) == 0:
        return np.nan
    ip, ib = boxes_gdf.sindex.query(polys.geometry, predicate='intersects')
    best = {}
    for i, j in zip(ip, ib):
        a = polys.geometry.iloc[i].intersection(boxes_gdf.geometry.iloc[j]).area
        if a > best.get(i, (0, -1))[0]:
            best[i] = (a, j)
    hit = 0
    for i, (_, j) in best.items():
        pb, bb = polys.geometry.iloc[i].bounds, boxes_gdf.geometry.iloc[j].bounds
        if min(pb[0] - bb[0], pb[1] - bb[1], bb[2] - pb[2], bb[3] - pb[3]) <= tol:
            hit += 1
    return round(100 * hit / len(polys), 1)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [ ]:
# @title 10. (необязательно) Подбор расширения рамок YOLOv9 на одном файле
# Прогоняет SAM на квадрате SWEEP_SIZE_M в центре снимка с разными вариантами расширения.
# Смотрите на картинки и колонку «упёрлись в рамку»: чем она ниже, тем реже рамка режет крону.
# Слишком большие рамки дают обратную проблему: SAM захватывает соседние кроны (растут диаметр и giant_pct).
RUN_SWEEP    = True
SWEEP_FILE   = FILES[0] if FILES else None   # на каком файле подбирать; можно указать путь вручную
SWEEP_SIZE_M = 100
SWEEP = [  # (BOX_SCALE, BOX_PAD_M, BOX_MIN_SIDE_M, BOX_SQUARE)
    (1.0, 0.0, 0.0, False),     # без расширения
    (1.2, 0.0, 0.0, False),
    (1.4, 0.0, 0.0, False),
    (1.0, 0.5, 0.0, False),
    (1.2, 0.0, 0.0, True),
]

if RUN_SWEEP and SWEEP_FILE:
    set_file(SWEEP_FILE)
    set_overview()
    GD_RAW, GD_SCORES, GD_TIME = geodeep_boxes()
    x0, y0, W, H = ROI
    side_px = int(SWEEP_SIZE_M / GSD)
    sroi = (x0 + max(0, (W - side_px) // 2), y0 + max(0, (H - side_px) // 2), min(side_px, W), min(side_px, H))
    sw_rows, sw_res = {}, {}
    for sc_, pad_, mn_, sq_ in SWEEP:
        name = f'sweep_s{sc_}_p{pad_}_m{mn_}' + ('_sq' if sq_ else '')
        bx = expand_boxes(GD_RAW, sc_, pad_, mn_, sq_)
        g_ = run_model(name, make_geodeep_sam_fn(bx, GD_SCORES), roi=sroi)
        inside = ((bx[:, 2] > sroi[0]) & (bx[:, 0] < sroi[0] + sroi[2]) &
                  (bx[:, 3] > sroi[1]) & (bx[:, 1] < sroi[1] + sroi[3]))
        bg = boxes_to_gdf(bx[inside], GD_SCORES[inside])
        a = g_.geometry.area if len(g_) else pd.Series([np.nan])
        sw_rows[name] = {
            'рамок': int(np.sum(inside)), 'крон': len(g_),
            'median_diam_m': round(float(2 * np.sqrt(a.median() / np.pi)), 2),
            'giant_pct': round(float(100 * (a > MAX_AREA_M2).mean()), 1),
            'compact_med': round(float((4 * np.pi * a / g_.geometry.length**2).median()), 3) if len(g_) else np.nan,
            'упёрлись в рамку, %': box_hit_pct(g_, bg),
            'время, с': round(g_.attrs['time_s']),
        }
        sw_res[name] = (g_, bg)
    display(pd.DataFrame(sw_rows).T)

    with rasterio.open(ORTHO) as src:
        win = Window(*sroi)
        img_s = src.read([1, 2, 3], window=win).transpose(1, 2, 0)
        ext_l, ext_t = src.window_transform(win) * (0, 0)
        ext_r, ext_b = src.window_transform(win) * (sroi[2], sroi[3])
    n = len(sw_res)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 6), squeeze=False)
    for ax, (k, (g_, bg)) in zip(axes.ravel(), sw_res.items()):
        ax.imshow(img_s, extent=(ext_l, ext_r, ext_b, ext_t))
        bg.boundary.plot(ax=ax, color=BOX_HEX, linewidth=0.7, linestyle='--')
        g_.boundary.plot(ax=ax, color=LINE_HEX, linewidth=1.2)
        ax.set_xlim(ext_l, ext_r); ax.set_ylim(ext_b, ext_t); ax.axis('off')
        ax.set_title(f'{k} ({len(g_)})')
    plt.tight_layout()
    plt.savefig(f'{OUT}/box_sweep.png', dpi=130)
    plt.show()
    print('Выбранный вариант впишите в BOX_SCALE / BOX_PAD_M / BOX_MIN_SIDE_M / BOX_SQUARE (ячейка 4),')
    print('перезапустите ячейку 4 и затем ячейку 17.')

In [ ]:
# @title 11. GeoDeep YOLOv9 → SAM 2.1 (функция для одного файла)
import gc, torch

BOX_GDFS = {}   # {связка: (исходные рамки, рамки для SAM)} текущего файла, для картинки крупного плана


def run_yolo_sam():
    global GD_RAW, GD_SCORES, GD_TIME, GD_BOXES
    GD_RAW, GD_SCORES, GD_TIME = geodeep_boxes()
    raw_gdf = boxes_to_gdf(GD_RAW, GD_SCORES)
    if SAVE['boxes_gpkg']:
        raw_gdf.to_file(f'{OUT}/geodeep_boxes_raw.gpkg', driver='GPKG')
    GD_BOXES = expand_boxes(GD_RAW, BOX_SCALE, BOX_PAD_M, BOX_MIN_SIDE_M, BOX_SQUARE)
    boxes_gdf = boxes_to_gdf(GD_BOXES, GD_SCORES)
    if SAVE['boxes_gpkg']:
        boxes_gdf.to_file(f'{OUT}/geodeep_boxes.gpkg', driver='GPKG')
    BOX_GDFS[MODEL_KEY] = (raw_gdf, boxes_gdf)
    if len(GD_RAW):
        print(f'Рамки YOLOv9: медианная сторона {np.median(side(GD_RAW)):.2f} -> {np.median(side(GD_BOXES)):.2f} м')
    g = run_model(MODEL_KEY, make_geodeep_sam_fn(GD_BOXES, GD_SCORES), extra_time=GD_TIME)
    print(f'{MODEL_KEY}: рамок {len(GD_BOXES)} -> крон {len(g)}, '
          f'упёрлись в рамку: {box_hit_pct(g, boxes_gdf)}%')
    gc.collect(); torch.cuda.empty_cache()
    return g

In [ ]:
# @title 11b. DeepForest → SAM 2.1 (функция для одного файла)
# Та же модель, что GeoDeep 'trees', но из первоисточника: ONNX-версия GeoDeep падает в onnxruntime
from deepforest import main as df_main

_DF = None
def get_deepforest():
    # Модель загружается один раз на все файлы
    global _DF
    if _DF is None:
        _DF = df_main.deepforest(config_args={'score_thresh': DF_SCORE})
        _DF.load_model(DF_MODEL)
        _DF.model.score_thresh = DF_SCORE
    return _DF


def deepforest_boxes(roi=None):
    # Рамки DeepForest по всему ROI в пикселях ORTHO: (boxes Nx4, scores N, время, с)
    roi = roi or ROI
    t0 = time.time()
    x0, y0, W, H = roi
    with rasterio.open(ORTHO) as src:
        img = src.read([1, 2, 3], window=Window(x0, y0, W, H)).transpose(1, 2, 0)   # RGB
    df = get_deepforest().predict_tile(image=np.ascontiguousarray(img),
                                       patch_size=DF_PATCH, patch_overlap=DF_OVERLAP)
    if df is None or len(df) == 0:
        boxes, scores = np.zeros((0, 4)), np.zeros(0)
    else:
        boxes = df[['xmin', 'ymin', 'xmax', 'ymax']].to_numpy(float) + [x0, y0, x0, y0]
        scores = df['score'].to_numpy(float)
    print(f'DeepForest: {len(boxes)} рамок, {time.time() - t0:.0f} с')
    return boxes, scores, time.time() - t0


def run_deepforest_sam():
    global RN_RAW, RN_SCORES, RN_TIME, RN_BOXES
    RN_RAW, RN_SCORES, RN_TIME = deepforest_boxes()
    raw_gdf = boxes_to_gdf(RN_RAW, RN_SCORES)
    if SAVE['boxes_gpkg']:
        raw_gdf.to_file(f'{OUT}/deepforest_boxes_raw.gpkg', driver='GPKG')
    RN_BOXES = expand_boxes(RN_RAW, *RN_BOX)
    boxes_gdf = boxes_to_gdf(RN_BOXES, RN_SCORES)
    if SAVE['boxes_gpkg']:
        boxes_gdf.to_file(f'{OUT}/deepforest_boxes.gpkg', driver='GPKG')
    BOX_GDFS[RN_KEY] = (raw_gdf, boxes_gdf)
    if len(RN_RAW):
        print(f'Рамки DeepForest: медианная сторона {np.median(side(RN_RAW)):.2f} -> {np.median(side(RN_BOXES)):.2f} м')
    g = run_model(RN_KEY, make_geodeep_sam_fn(RN_BOXES, RN_SCORES), extra_time=RN_TIME)
    print(f'{RN_KEY}: рамок {len(RN_BOXES)} -> крон {len(g)}, '
          f'упёрлись в рамку: {box_hit_pct(g, boxes_gdf)}%')
    gc.collect(); torch.cuda.empty_cache()
    return g

## Результаты (функции)

In [ ]:
# @title 12. Таблица метрик (функция для одного файла)
def box_label(scale, pad, square):
    return f'рамки x{scale}' + (f' +{pad} м' if pad else '') + (', квадрат' if square else '')


def flags(r):
    f = []
    if r['giant_pct'] > 5:              f.append('много слипшихся крон (> MAX_AREA_M2)')
    if r['self_overlap'] > 0.10:        f.append('дубли или наложения')
    if r['p5_area'] < 0.05 * r['median_area']: f.append('много мелкого мусора')
    if r['compact_med'] < 0.5:          f.append('рваные/вытянутые контуры')
    if r['veg_covered_pct'] < 30:       f.append('пропускает много «зелёного»')
    if r['veg_in_poly_pct'] < 50:       f.append('много полигонов вне «зелёного» (для сосны ExG ненадёжен)')
    return '; '.join(f) or 'явных проблем не видно'


def report_metrics(results, show=True):
    # Метрики без разметки для всех связок текущего файла; сохраняет model_summary.csv в OUT
    models = {k: v for k, v in results.items() if len(v) > 0}
    for k in set(results) - set(models):
        print('Без результатов (0 крон):', k)
    if not models:
        return None
    MODEL_INFO = {
        MODEL_KEY: dict(Метод='GeoDeep YOLOv9 -> SAM 2.1, ' + box_label(BOX_SCALE, BOX_PAD_M, BOX_SQUARE),
                        Тип='детекция + сегментация по рамке'),
        RN_KEY:    dict(Метод='DeepForest RetinaNet -> SAM 2.1, ' + box_label(RN_BOX[0], RN_BOX[1], RN_BOX[3]),
                        Тип='детекция + сегментация по рамке'),
    }
    info_df = pd.DataFrame({k: MODEL_INFO.get(k, dict(Метод='—', Тип='—')) for k in models}).T

    roi_ha = valid.sum() * abs(tr.a * tr.e) / 1e4
    print(f'Площадь снимка: {roi_ha:.2f} га')
    rows = {}
    for k, g in models.items():
        m = proxy_metrics(k, g)
        m['area_ha'] = round(roi_ha, 2)
        m['crowns_per_ha'] = round(m['n_crowns'] / roi_ha, 1)
        m['sec_per_ha'] = round(m['time_s'] / roi_ha, 2) if np.isfinite(m['time_s']) else np.nan
        m['median_diam_m'] = round(2 * np.sqrt(m['median_area'] / np.pi), 2)
        m['box_hit_pct'] = box_hit_pct(g, BOX_GDFS[k][1]) if k in BOX_GDFS else np.nan
        rows[k] = m
    test_df = pd.DataFrame(rows).T
    if len(models) > 1:
        names = list(models)
        agree = pd.DataFrame([[agreement(a, models[a], b, models[b]) if a != b else np.nan for b in names]
                              for a in names], index=names, columns=names)
        test_df['mean_agreement'] = agree.mean(axis=1).round(3)

    cols = ['area_ha', 'n_crowns', 'crowns_per_ha', 'median_diam_m', 'median_area', 'p5_area', 'p95_area',
            'giant_pct', 'compact_med', 'self_overlap', 'box_hit_pct', 'cover_pct', 'exg_in', 'exg_out',
            'veg_in_poly_pct', 'veg_covered_pct', 'mean_agreement', 'time_s', 'sec_per_ha']
    test_df = test_df[[c for c in cols if c in test_df.columns]].apply(pd.to_numeric, errors='coerce')
    summary = info_df.join(test_df)
    summary['Замечания'] = [flags(test_df.loc[k]) for k in summary.index]
    if show:
        display(test_df)
    if SAVE['summary_csv']:
        summary.to_csv(f'{OUT}/model_summary.csv', encoding='utf-8-sig')
    if SAVE['summary_html']:
        summary.to_html(f'{OUT}/model_summary.html', escape=False)
    return summary

In [ ]:
# @title 13. Крупный план: рамки и контуры SAM (функция для одного файла)
import matplotlib.pyplot as plt


def plot_closeup(results, show=True):
    models = {k: v for k, v in results.items() if len(v) > 0}
    if not models or not (SAVE['closeup_png'] or show):
        return
    h_px, w_px = arr.shape[1:]
    left, top = tr.c, tr.f
    right, bottom = tr.c + tr.a * w_px, tr.f + tr.e * h_px
    cx, cy = (left + right) / 2, (top + bottom) / 2
    half = VIS_SIZE_M / 2
    rgb = np.clip(arr.transpose(1, 2, 0) / 255, 0, 1)

    fig, axes = plt.subplots(len(models), 2, figsize=(12, 6 * len(models)), squeeze=False)
    for row, (k, g) in zip(axes, models.items()):
        for ax in row:
            ax.imshow(rgb, extent=(left, right, bottom, top))
            ax.set_xlim(cx - half, cx + half); ax.set_ylim(cy - half, cy + half); ax.axis('off')
        raw_gdf, boxes_gdf = BOX_GDFS.get(k, (None, None))
        if raw_gdf is not None:
            raw_gdf.boundary.plot(ax=row[0], color=BOX_HEX, linewidth=1.2)
            boxes_gdf.boundary.plot(ax=row[0], color='white', linewidth=0.8, linestyle='--')
            boxes_gdf.boundary.plot(ax=row[1], color=BOX_HEX, linewidth=0.6, linestyle='--')
        row[0].set_title(f'{k}: рамки ({len(raw_gdf) if raw_gdf is not None else 0}), пунктир = для SAM')
        if HALO_HEX:
            g.boundary.plot(ax=row[1], color=HALO_HEX, linewidth=2.2)
        g.boundary.plot(ax=row[1], color=LINE_HEX, linewidth=1.2)
        row[1].set_title(f'{k}: кроны ({len(g)})')
    fig.suptitle(FILE_NAME)
    plt.tight_layout()
    if SAVE['closeup_png']:
        plt.savefig(f'{OUT}/visual_closeup.png', dpi=150)
    plt.show() if show else plt.close(fig)

In [ ]:
# @title 14. Оценка по эталонной ручной разметке (функция для одного файла)
def evaluate(pred, ref, plots_union, thr=IOU_THR):
    p = pred[pred.geometry.centroid.within(plots_union)].reset_index(drop=True)
    r = ref[ref.geometry.centroid.within(plots_union)].reset_index(drop=True)
    pa, ra = p.geometry.area.values, r.geometry.area.values
    pairs = []
    if len(p) and len(r):
        ip, ir = r.sindex.query(p.geometry, predicate='intersects')
        for i, j in zip(ip, ir):
            inter = p.geometry.iloc[i].intersection(r.geometry.iloc[j]).area
            iou = inter / (pa[i] + ra[j] - inter)
            if iou >= thr:
                pairs.append((iou, i, j))
    used_p, used_r, match = set(), set(), []
    for iou, i, j in sorted(pairs, reverse=True):      # жадное сопоставление один к одному
        if i in used_p or j in used_r: continue
        used_p.add(i); used_r.add(j); match.append((iou, i, j))
    tp = len(match); fp = len(p) - tp; fn = len(r) - tp
    prec = tp / max(len(p), 1); rec = tp / max(len(r), 1)
    res = dict(n_ref=len(r), n_pred=len(p), TP=tp, FP=fp, FN=fn,
               precision=round(prec, 3), recall=round(rec, 3),
               F1=round(2 * prec * rec / max(prec + rec, 1e-9), 3))
    if match:
        ious = np.array([m[0] for m in match])
        a_p = np.array([pa[m[1]] for m in match]); a_r = np.array([ra[m[2]] for m in match])
        d_p, d_r = 2 * np.sqrt(a_p / np.pi), 2 * np.sqrt(a_r / np.pi)
        res.update(mean_IoU=round(float(ious.mean()), 3),
                   area_err_med_pct=round(float(100 * np.median(np.abs(a_p - a_r) / a_r)), 1),
                   area_bias_pct=round(float(100 * np.median((a_p - a_r) / a_r)), 1),
                   diam_rmse_m=round(float(np.sqrt(np.mean((d_p - d_r) ** 2))), 2))
    return res

def eval_reference(results):
    # Оценка по эталону текущего файла: REF_DIR/<имя файла>_ref_crowns.gpkg и _ref_plots.gpkg
    ref_gpkg = f'{REF_DIR}/{FILE_NAME}_ref_crowns.gpkg'
    plots_gpkg = f'{REF_DIR}/{FILE_NAME}_ref_plots.gpkg'
    models = {k: v for k, v in results.items() if len(v) > 0}
    if not models or not (os.path.exists(ref_gpkg) and os.path.exists(plots_gpkg)):
        print('Эталона для этого файла нет, оценка пропущена')
        return None
    crs = next(iter(models.values())).crs
    ref = gpd.read_file(ref_gpkg).to_crs(crs)
    plots = gpd.read_file(plots_gpkg).to_crs(crs)
    pu = plots.geometry.union_all() if hasattr(plots.geometry, 'union_all') else plots.geometry.unary_union
    print(f'Эталон: {len(ref)} крон на {plots.geometry.area.sum() / 1e4:.2f} га')
    eval_df = pd.DataFrame({k: evaluate(g, ref, pu) for k, g in models.items()}).T
    display(eval_df)
    eval_df.to_csv(f'{OUT}/eval_vs_reference.csv', encoding='utf-8-sig')
    return eval_df

In [ ]:
# @title 15. Результат целиком (PNG, синие контуры; функция для одного файла)
CROP_TO_DATA = True       # обрезать пустые поля вокруг ортофото
BG_MODE      = 'white'    # 'white' или 'transparent'


def overlay_base():
    # Подложка и рамка обрезки для текущего файла
    global base, cy0, cy1, cx0, cx1
    base = np.clip(arr.transpose(1, 2, 0), 0, 255).astype(np.uint8)
    if CROP_TO_DATA and valid.any():
        ys, xs = np.where(valid)
        cy0, cy1, cx0, cx1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    else:
        cy0, cy1, cx0, cx1 = 0, valid.shape[0], 0, valid.shape[1]


def render_overlay(gdf):
    img = base.copy()
    if len(gdf):
        bnd = rasterize(((g, 1) for g in gdf.geometry.boundary if not g.is_empty),
                        out_shape=exg.shape, transform=tr, fill=0, all_touched=True).astype(np.uint8)
        line = cv2.dilate(bnd, np.ones((LINE_PX, LINE_PX), np.uint8)) if LINE_PX > 1 else bnd
        if HALO_COLOR:
            k = LINE_PX + 2 * HALO_PX
            halo = cv2.dilate(bnd, np.ones((k, k), np.uint8))
            img[halo > 0] = HALO_COLOR
        img[line > 0] = LINE_COLOR
    if BG_MODE == 'transparent':
        img = np.dstack([img, (valid * 255).astype(np.uint8)])
    else:
        img[~valid] = 255
    return img[cy0:cy1, cx0:cx1]

def save_full_png(results):
    if not SAVE['full_png']:
        return
    overlay_base()
    for k, g in results.items():
        if len(g) == 0:
            print('Пропущено (0 крон):', k)
            continue
        img = render_overlay(g)
        p = f'{OUT}/full_{k}.png'
        cv2.imwrite(p, cv2.cvtColor(img, cv2.COLOR_RGBA2BGRA if img.shape[2] == 4 else cv2.COLOR_RGB2BGR))
        print('Сохранено:', p, img.shape[:2])

In [ ]:
# @title 16. Экспорт на проверку в QGIS (функция для одного файла)
MIN_COMPACT = 0.45         # 4*pi*A/P^2: 1 = круг, малые значения = вытянутые/рваные формы


def export_review(k, gdf):
    g = gdf.copy()
    g['id'] = np.arange(1, len(g) + 1)
    g['area_m2'] = g.geometry.area.round(2)
    g['diam_m'] = (2 * np.sqrt(g.geometry.area / np.pi)).round(2)
    g['compact'] = (4 * np.pi * g.geometry.area / g.geometry.length**2).round(3)
    reason = np.where(g.area_m2 < MIN_AREA_M2, 'мелкий',
             np.where(g.area_m2 > MAX_AREA_M2, 'слишком крупный',
             np.where(g.compact < MIN_COMPACT, 'неправильная форма', '')))
    g['reason'] = reason
    g['keep'] = (reason == '').astype(int)      # 1 = оставить, 0 = проверить/исключить
    review = f'{OUT}/{k}_review.gpkg'
    g[['id', 'keep', 'reason', 'area_m2', 'diam_m', 'compact', 'score', 'geometry']].to_file(review, driver='GPKG')
    print(f'{k}: всего {len(g)}, keep=1: {g.keep.sum()}, на проверку: {(g.keep == 0).sum()}')
    print('  файл для QGIS:', review, '| рамки:', BOX_FILES.get(k, '—') if SAVE['boxes_gpkg'] else 'не сохранялись')

def export_reviews(results):
    global BOX_FILES
    if not SAVE['review_gpkg']:
        return
    BOX_FILES = {MODEL_KEY: f'{OUT}/geodeep_boxes.gpkg', RN_KEY: f'{OUT}/deepforest_boxes.gpkg'}
    for k, g in results.items():
        if len(g) == 0:
            print('Пропущено (0 крон):', k)
            continue
        export_review(k, g)

## Запуск

In [ ]:
# @title 17. Обработка всех файлов
import traceback

SHOW_PLOTS = False   # True = показывать таблицы и крупный план по каждому файлу (на Диск сохраняются в любом случае)

all_summary, all_eval, failed = [], [], {}
for i, path in enumerate(FILES, 1):
    print('=' * 90)
    print(f'[{i}/{len(FILES)}] {os.path.basename(path)}')
    try:
        set_file(path)
        set_overview()
        BOX_GDFS.clear()
        results = {}
        if RUN_YOLO_SAM:
            results[MODEL_KEY] = run_yolo_sam()
        if RUN_DF_SAM:
            results[RN_KEY] = run_deepforest_sam()
        summ = report_metrics(results, show=SHOW_PLOTS)
        if summ is not None:
            all_summary.append(summ.rename_axis('модель').reset_index().assign(файл=FILE_NAME))
        plot_closeup(results, show=SHOW_PLOTS)
        ev = eval_reference(results)
        if ev is not None:
            all_eval.append(ev.rename_axis('модель').reset_index().assign(файл=FILE_NAME))
        save_full_png(results)
        export_reviews(results)
    except Exception as e:
        failed[path] = repr(e)
        print('ОШИБКА, файл пропущен:', e)
        traceback.print_exc()
    finally:
        plt.close('all'); gc.collect(); torch.cuda.empty_cache()
        if not SAVE['keep_prepared'] and 'ORTHO' in globals() and os.path.exists(ORTHO) \
                and os.path.abspath(ORTHO).startswith(os.path.abspath(PREP_DIR)):
            os.remove(ORTHO)                     # удаляем только копию из PREP_DIR, исходник не трогаем
            print('Удалена подготовленная копия:', ORTHO)

print('=' * 90)
if all_summary:
    total = pd.concat(all_summary, ignore_index=True)
    total = total[['файл'] + [c for c in total.columns if c != 'файл']]
    total.to_csv(f'{OUT_ROOT}/all_files_summary.csv', encoding='utf-8-sig', index=False)
    print('Сводка по всем файлам:', f'{OUT_ROOT}/all_files_summary.csv')
    display(total[['файл', 'модель', 'area_ha', 'n_crowns', 'crowns_per_ha', 'median_diam_m',
                   'giant_pct', 'box_hit_pct', 'cover_pct', 'sec_per_ha']])
    print('Среднее по файлам:')
    display(total.groupby('модель')[['crowns_per_ha', 'median_diam_m', 'giant_pct', 'compact_med',
                                     'box_hit_pct', 'cover_pct', 'sec_per_ha']].mean().round(2))
if all_eval:
    total_eval = pd.concat(all_eval, ignore_index=True)
    total_eval.to_csv(f'{OUT_ROOT}/all_files_eval.csv', encoding='utf-8-sig', index=False)
    print('Оценка по эталону:', f'{OUT_ROOT}/all_files_eval.csv')
    display(total_eval)
if failed:
    print('Файлы с ошибками:')
    for p, e in failed.items():
        print(' ', os.path.basename(p), '->', e)

[1/1] Ортофотоплан (Бульвар Победы).tif
Уже существует, пропускаю: /content/drive/MyDrive/tree_model_test/ortho_prepared/Ортофотоплан (Бульвар Победы)_utm_10cm.tif
EPSG:32636 (0.1, 0.1) 3199 4143 4 ('uint8', 'uint8', 'uint8', 'uint8')
Ортофотоплан (Бульвар Победы): GSD=0.100 м | площадь кроны 200-8000 пикс | результаты: /content/drive/MyDrive/tree_model_test/results_multi/Ортофотоплан (Бульвар Победы)
Порог ExG (Оцу): 0.0952
GeoDeep: 66 рамок, 162 с
Рамки YOLOv9: медианная сторона 7.03 -> 8.44 м


geodeep_sam:   0%|          | 0/24 [00:00<?, ?it/s]

geodeep_sam: 61 крон, 170 с
geodeep_sam: рамок 66 -> крон 61, упёрлись в рамку: 55.7%
Площадь снимка: 10.35 га
Эталона для этого файла нет, оценка пропущена
Сохранено: /content/drive/MyDrive/tree_model_test/results_multi/Ортофотоплан (Бульвар Победы)/full_geodeep_sam.png (2966, 2286)
Сводка по всем файлам: /content/drive/MyDrive/tree_model_test/results_multi/all_files_summary.csv


,файл,модель,area_ha,n_crowns,crowns_per_ha,median_diam_m,giant_pct,box_hit_pct,cover_pct,sec_per_ha
0,Ортофотоплан (Бульвар Победы),geodeep_sam,10.35,61.0,5.9,6.5,0.0,55.7,2.2,16.42


Среднее по файлам:


,crowns_per_ha,median_diam_m,giant_pct,compact_med,box_hit_pct,cover_pct,sec_per_ha
модель,,,,,,,
geodeep_sam,5.9,6.5,0.0,0.7,55.7,2.2,16.42
